In [1]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

from transformers import ViTModel
import matplotlib.pyplot as plt
from tqdm import tqdm
import os

d:\Users\H P\Desktop\sem7\CV\SinhalaOCR\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Matplotlib is building the font cache; this may take a moment.


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
DATASET_PATH = PROJECT_ROOT / "Dataset454"
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 454
EPOCHS = 2   # NOTE: notebook only for testing
LR = 3e-4

if not DATASET_PATH.exists():
    raise FileNotFoundError(f"Dataset folder not found: {DATASET_PATH}")

device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(5),
    transforms.RandomAffine(0, translate=(0.05, 0.05)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

In [5]:
train_dataset = ImageFolder(
    root=os.path.join(DATASET_PATH, "train"),
    transform=train_transform
)

val_dataset = ImageFolder(
    root=os.path.join(DATASET_PATH, "val"),
    transform=test_transform
)

test_dataset = ImageFolder(
    root=os.path.join(DATASET_PATH, "test"),
    transform=test_transform
)

print("Classes:", len(train_dataset.classes))

FileNotFoundError: [WinError 3] The system cannot find the path specified: 'Dataset454\\train'

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
images, labels = next(iter(train_loader))

plt.figure(figsize=(10,5))
for i in range(8):
    plt.subplot(2,4,i+1)
    plt.imshow(images[i].permute(1,2,0))
    plt.title(labels[i].item())
    plt.axis("off")
plt.show()

In [ ]:
class SinhalaViT(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = ViTModel.from_pretrained(
            "facebook/deit-base-patch16-224"
        )

        hidden = self.encoder.config.hidden_size
        self.classifier = nn.Linear(hidden, NUM_CLASSES)

    def forward(self, x):
        x = self.encoder(pixel_values=x).last_hidden_state[:, 0]
        x = self.classifier(x)
        return x

model = SinhalaViT().to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

In [ ]:
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}")

    for imgs, labels in loop:
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(imgs)

        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    print(f"Epoch {epoch+1} Loss:", total_loss / len(train_loader))

In [ ]:
def evaluate():
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)

            outputs = model(imgs)
            preds = torch.argmax(outputs, dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    acc = 100 * correct / total
    return acc


print("Validation Accuracy:", evaluate())

In [ ]:
model.eval()

img, label = val_dataset[0]
img = img.unsqueeze(0).to(device)

with torch.no_grad():
    pred = model(img)

print("Pred:", torch.argmax(pred), "Actual:", label)

In [ ]:
torch.save(model.state_dict(), "vit_sinhala_454.pth")